<a href="https://colab.research.google.com/github/Ahmed-Hamam/control84up224/blob/main/R84up224.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CUB-200-2011 — MobileNetV3-SMALL — CONTROL 84→224 (upsampled) — COLAB + GITHUB VERSION
# ============================================================================
# نسخة Colab من سكريبت الـ control (Stage 1 + Stage 2 كاملين، seed 1 افتراضيًا):
# الصورة: Resize 96 → crop 84 → تكبير bilinear لـ 224 → normalization.
# بيحفظ كل حاجة (progress_control.json + checkpoints) على GitHub بعد كل seed/eval،
# ولما تشغّل في جلسة Colab جديدة بيسترجع آخر نسخة من GitHub ويكمّل.
#
# قبل التشغيل في Colab (مرة في كل جلسة):
#   1) Runtime → Change runtime type → GPU.
#   2) Secrets (أيقونة المفتاح في الشريط الجانبي) → ضيف GITHUB_TOKEN
#      (GitHub PAT بصلاحية repo) وفعّل "Notebook access".
#      لتنزيل الداتاست من Kaggle: إمّا Secrets باسم KAGGLE_USERNAME و KAGGLE_KEY،
#      أو ارفع kaggle.json في /root/.kaggle/ زي ما كنت بتعمل.
#   3) غيّر GITHUB_REPO تحت لمستودعك. ⚠️ لازم نفس GITHUB_REPO و GITHUB_SUBFOLDER
#      بتوع تجربة Small الأساسية (84/224) عشان المقارنة الثلاثية تشتغل.
#
# ملاحظات: Colab المجاني بيقفل الجلسة بعد ساعات → الـ push بعد كل seed/eval هو اللي
# بيحمي شغلك. الـ checkpoint الوسيط جوه Stage 1/2 مش بيتحفظ، فلو الجلسة اتقطعت في
# نص seed، الـ seed ده بيتعاد من أوله.
# ============================================================================

import os
import json
import glob
import shutil
import subprocess

try:
    import kagglehub
except ImportError:
    print("📦 Installing kagglehub...")
    os.system("pip install -q kagglehub")
    import kagglehub


def _colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


def _get_github_token():
    return os.environ.get("GITHUB_TOKEN") or _colab_secret("GITHUB_TOKEN")


# مصادقة Kaggle (لو متاحة من Colab Secrets) عشان kagglehub ينزّل الداتاست
for _k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
    if not os.environ.get(_k):
        _v = _colab_secret(_k)
        if _v:
            os.environ[_k] = _v

try:
    from thop import profile as thop_profile
except ImportError:
    print("📦 Installing thop...")
    os.system('pip install -q thop')
    from thop import profile as thop_profile

try:
    from scipy import stats as scipy_stats
except ImportError:
    print("📦 Installing scipy...")
    os.system('pip install -q scipy')
    from scipy import stats as scipy_stats

import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from tqdm import tqdm
from PIL import Image

# ============================================
# 0. PATHS (Kaggle: input للداتاست، working للتدريب/الحفظ)
# ============================================
# الداتاست: من /kaggle/input مباشرة (مفيش تنزيل خالص)
KAGGLE_INPUT_ROOT = "/nonexistent_kaggle_input"   # مش مستخدم على Colab
CUB_KAGGLEHUB_ID  = "wenewone/cub2002011"        # عدّله لو داتاست CUB عندك مختلف

# مكان العمل: /kaggle/working (بيتحفظ لما تعمل Save Version)
WORK_ROOT      = "/content/cub200_small_workdir"   # ⚠️ نفس اسم تجربة Small الأساسية لو مختلف
DATA_ROOT      = "/content/cub200_data_local"
CUB_SPLIT_ROOT = os.path.join(DATA_ROOT, "cub200_split")
os.makedirs(WORK_ROOT, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

print(f"📂 WORK_ROOT (progress + checkpoints, بيتحفظ مع Save Version) = {WORK_ROOT}")

# 🆕 إعداد GitHub Auto-Push — غيّر القيمة دي لاسم مستودعك (username/reponame)
GITHUB_REPO       = "your-username/cub200-small-results"  # ⚠️ نفس مستودع تجربة Small الأساسية  # ⚠️ غيّرها!
GITHUB_BRANCH     = "main"
GITHUB_CLONE_DIR  = "/content/_github_results_repo"
GITHUB_SUBFOLDER  = "cub200_small_results"  # ⚠️ نفس المجلد الفرعي بتاع تجربة Small الأساسية  # المجلد جوه المستودع اللي هتتحط فيه الملفات


def _run_git_cmd(args, cwd=None, env=None):
    """ينفّذ أمر git ويرجّع (success, output). بيلتقط stderr/stdout في
    الرسالة نفسها عشان نقدر نطبع سبب الفشل لو حصل. (معرّفة هنا فوق عشان
    تُستخدم في الاسترجاع التلقائي من GitHub بالأول، وكمان في push_results_to_github تحت)."""
    result = subprocess.run(
        ["git"] + args, cwd=cwd, env=env,
        capture_output=True, text=True
    )
    ok = (result.returncode == 0)
    output = (result.stdout or "") + (result.stderr or "")
    return ok, output


def _restore_previous_workdir_from_input():
    """🆕 يدوّر في /kaggle/input على أي مجلد مضاف فيه progress.json (يعني
    ده Output بتاع نسخة سابقة من نفس النوتبوك أُضيف كـ Input)، ولو لقاه
    بينسخ كل محتوياته لـ WORK_ROOT محليًا (نسخ محلي سريع، مش عبر الشبكة)
    عشان الـ Resume يشتغل عادي."""
    if os.path.exists(os.path.join(WORK_ROOT, "progress.json")):
        print("✅ progress.json موجود بالفعل في WORK_ROOT — مفيش حاجة نسترجعها.")
        return

    candidates = glob.glob(os.path.join(KAGGLE_INPUT_ROOT, "*", "**", "progress.json"),
                            recursive=True)
    if not candidates:
        print("ℹ️  مفيش نسخة سابقة (progress.json) موجودة في أي Input — هنبدأ من الصفر.")
        return

    # لو فيه أكتر من واحد (مش متوقع)، ناخد أحدث واحد بالـ mtime
    src_progress = max(candidates, key=os.path.getmtime)
    src_dir = os.path.dirname(src_progress)
    print(f"♻️  لقينا نسخة سابقة في: {src_dir}")
    print("📦 بننسخ progress.json وكل الـ checkpoints لـ WORK_ROOT محليًا...")

    for item in os.listdir(src_dir):
        src_path = os.path.join(src_dir, item)
        dst_path = os.path.join(WORK_ROOT, item)
        if os.path.isfile(src_path) and not os.path.exists(dst_path):
            shutil.copy2(src_path, dst_path)

    print(f"✅ تم استرجاع {len(os.listdir(WORK_ROOT))} ملف من النسخة السابقة.")


def _restore_previous_workdir_from_github():
    """🆕 لو مفيش نسخة سابقة في /kaggle/input (يعني أول تشغيل في notebook
    جديد، أو نسيت تضيف الـ Input يدويًا)، نحاول نسحب آخر progress.json +
    checkpoints مباشرة من مستودع GitHub (لو GITHUB_REPO متظبط ومتاح الـ
    GITHUB_TOKEN). ده بيخلي الاستمرارية تلقائية بالكامل، من غير ما تحتاج
    تضيف Input يدوي كل مرة."""
    if os.path.exists(os.path.join(WORK_ROOT, "progress.json")):
        return  # اتسترجعت بالفعل من /kaggle/input

    if GITHUB_REPO.startswith("your-username/"):
        print("ℹ️  GITHUB_REPO لسه مش معدّل — هنتجاهل الاسترجاع التلقائي من GitHub.")
        return

    token = _get_github_token()
    if not token:
        print("ℹ️  مقدرتش أجيب GITHUB_TOKEN (Colab Secrets) — هنتجاهل الاسترجاع "
              "التلقائي من GitHub ونبدأ من الصفر.")
        return

    print(f"\n🔍 بنحاول نسترجع نسخة سابقة من مستودع GitHub ({GITHUB_REPO})...")
    remote_url = f"https://{token}@github.com/{GITHUB_REPO}.git"

    try:
        if not os.path.exists(os.path.join(GITHUB_CLONE_DIR, ".git")):
            ok, out = _run_git_cmd([
                "clone", "--depth", "1", "--branch", GITHUB_BRANCH,
                remote_url, GITHUB_CLONE_DIR
            ])
            if not ok:
                print(f"ℹ️  مقدرناش نعمل clone للمستودع (غالبًا فاضي أو أول مرة) "
                      f"— هنبدأ من الصفر:\n{out}")
                return
        else:
            ok, out = _run_git_cmd(["pull", "--rebase", "origin", GITHUB_BRANCH],
                                    cwd=GITHUB_CLONE_DIR)

        src_dir = os.path.join(GITHUB_CLONE_DIR, GITHUB_SUBFOLDER)
        src_progress = os.path.join(src_dir, "progress.json")
        if not os.path.exists(src_progress):
            print("ℹ️  مفيش progress.json جوه المستودع — هنبدأ من الصفر.")
            return

        print(f"♻️  لقينا نسخة سابقة على GitHub في {src_dir}")
        print("📦 بننسخ progress.json وكل الـ checkpoints لـ WORK_ROOT محليًا...")
        for item in os.listdir(src_dir):
            src_path = os.path.join(src_dir, item)
            dst_path = os.path.join(WORK_ROOT, item)
            if os.path.isfile(src_path) and not os.path.exists(dst_path):
                shutil.copy2(src_path, dst_path)

        print(f"✅ تم استرجاع {len(os.listdir(WORK_ROOT))} ملف من GitHub — "
              f"الكود هيكمّل من عند آخر seed اتسجّل بدل ما يبدأ من الصفر.")

    except Exception as e:
        print(f"⚠️  الاسترجاع التلقائي من GitHub فشل بشكل غير متوقع ({e}) "
              f"— هنبدأ من الصفر.")


_restore_previous_workdir_from_github()

# ============================================
# 1. SETUP
# ============================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔥 Processing on: {device}")

STAGE_TIMES = {}

class StageTimer:
    """Context manager that records wall-clock time for a named stage."""
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.time()
        print(f"\n⏱️  [{self.name}] started...")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        elapsed = time.time() - self.t0
        STAGE_TIMES[self.name] = elapsed
        mins = elapsed / 60.0
        print(f"⏱️  [{self.name}] finished in {elapsed:.1f}s ({mins:.2f} min)")


def print_stage_time_summary():
    if not STAGE_TIMES:
        print("No stage timings recorded yet.")
        return
    print("\n" + "="*40)
    print("⏱️  STAGE TIME SUMMARY")
    print("="*40)
    total = 0.0
    for name, secs in STAGE_TIMES.items():
        print(f"  {name:45s} {secs:8.1f}s  ({secs/60:.2f} min)")
        total += secs
    print(f"  {'TOTAL':45s} {total:8.1f}s  ({total/60:.2f} min)")

# ============================================
# 2. MODEL PROFILING: PARAMETERS & FLOPs
# ============================================

def count_parameters(model, trainable_only=False):
    if trainable_only:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    return sum(p.numel() for p in model.parameters())


def compute_flops(backbone_module, input_size, device=device):
    """Backbone-only FLOPs/params via thop (feature extraction pass only)."""
    backbone_module = backbone_module.to(device).eval()
    dummy = torch.randn(1, 3, input_size, input_size).to(device)

    class _FeatureOnly(nn.Module):
        def __init__(self, backbone):
            super().__init__()
            self.backbone = backbone

        def forward(self, x):
            x = self.backbone.features(x)
            x = self.backbone.avgpool(x)
            return torch.flatten(x, 1)

    wrapper = _FeatureOnly(backbone_module).to(device).eval()
    with torch.no_grad():
        macs, params = thop_profile(wrapper, inputs=(dummy,), verbose=False)

    gflops = macs / 1e9
    params_m = params / 1e6
    print(f"📐 Backbone profile @ {input_size}x{input_size}: "
          f"{params_m:.2f}M params | {gflops:.4f} GFLOPs (MACs)")
    return {"params": params, "params_M": params_m, "macs": macs, "gflops": gflops}


def report_full_model_stats(model, input_size):
    """Total params of ProtoNet wrapper (backbone only, no extra params) + FLOPs."""
    total_params = count_parameters(model)
    trainable_params = count_parameters(model, trainable_only=True)
    print(f"\n📊 ProtoNet full model: "
          f"{total_params/1e6:.3f}M total params | "
          f"{trainable_params/1e6:.3f}M trainable params")
    flops_info = compute_flops(model.backbone, input_size=input_size)
    return {
        "total_params": total_params,
        "trainable_params": trainable_params,
        **flops_info
    }

# ============================================
# 3. DATA PREPARATION — CUB-200-2011 (من /kaggle/input مباشرة)
# ============================================

def _find_images_folder(root):
    """أبسط وأوثق طريقة: يدوّر تحديدًا على مجلد اسمه 'images' (زي بنية
    CUB_200_2011/images/ الرسمية) وفيه مجلدات فرعية (الأصناف)."""
    for r, dirs, files in os.walk(root):
        if os.path.basename(r) == "images" and any(
            os.path.isdir(os.path.join(r, d)) for d in dirs
        ):
            return r
    return None


def _find_class_dirs(root):
    """Return a directory whose immediate children are all class-folders
    (heuristic: at least 50 subdirs, each containing images)."""
    for r, dirs, files in os.walk(root):
        if not dirs:
            continue
        sub_dirs = [d for d in dirs if os.path.isdir(os.path.join(r, d))]
        if len(sub_dirs) < 50:  # CUB has 200 classes total / per split >=50
            continue
        sample = os.path.join(r, sub_dirs[0])
        has_imgs = any(f.lower().endswith(('jpg', 'jpeg', 'png'))
                       for f in os.listdir(sample))
        if has_imgs:
            return r
    return None


def _find_cub_dataset_root():
    """Colab: بننزّل الداتاست من Kaggle بـ kagglehub (كاش محلي في /root/.cache)."""
    print(f"📥 Downloading CUB-200-2011 via kagglehub ({CUB_KAGGLEHUB_ID})...")
    try:
        return kagglehub.dataset_download(CUB_KAGGLEHUB_ID)
    except Exception as e:
        raise RuntimeError(
            f"❌ تنزيل الداتاست فشل ({e}). اتأكد من مصادقة Kaggle "
            "(Secrets: KAGGLE_USERNAME + KAGGLE_KEY، أو kaggle.json في /root/.kaggle/) "
            "وإن Internet شغّال.")


def _split_is_complete(split_root):
    """🆕 يتأكد إن الثلاثة فولدرات train/val/test موجودين فعلًا وفيهم
    أصناف (مش مجرد إن split_root نفسه موجود). بدون الفحص ده، لو حصل أي
    قطع أو مشكلة ديسك في نص بناء الداتاست (مثلًا train وval خلصوا وtest
    اتقطع)، الكود كان بيفتكر إن الداتاست 'جاهز' غلط ويكمّل بمجلد ناقص،
    وده كان سبب FileNotFoundError على مجلد test وقت الـ Matched-Episode
    Evaluation."""
    for split_name in ("train", "val", "test"):
        split_dir = os.path.join(split_root, split_name)
        if not os.path.isdir(split_dir):
            return False
        class_dirs = [d for d in os.listdir(split_dir)
                      if os.path.isdir(os.path.join(split_dir, d))]
        if not class_dirs:
            return False
    return True


def prepare_cub200():
    if os.path.exists(CUB_SPLIT_ROOT):
        if _split_is_complete(CUB_SPLIT_ROOT):
            print("✅ CUB-200-2011 split already present and looks complete "
                  "(train/val/test كلهم موجودين وفيهم أصناف).")
            return
        print("⚠️  لقينا مجلد split موجود لكنه ناقص (train/val/test مش كلهم "
              "كاملين — غالبًا بسبب انقطاع سابق في منتصف البناء). "
              "هنمسحه ونعيد بناء الداتاست بالكامل من تنزيل جديد.")
        shutil.rmtree(CUB_SPLIT_ROOT)

    print("\n🔍 بنجهّز داتاست CUB-200-2011...")
    dataset_path = _find_cub_dataset_root()
    if dataset_path is None:
        raise RuntimeError(
            "❌ مش لاقي داتاست CUB-200-2011."
        )
    print(f"📁 Dataset found at: {dataset_path}")

    try:
        os.makedirs(CUB_SPLIT_ROOT, exist_ok=True)

        # الحالة 1: الداتاست موزّع بالفعل بمجلدات train/val/test رسمية
        existing_splits = {}
        for split_name in ["train", "val", "test"]:
            for r, dirs, files in os.walk(dataset_path):
                if os.path.basename(r).lower() == split_name and dirs:
                    existing_splits[split_name] = r
                    break

        if len(existing_splits) == 3:
            print("📦 Found official train/val/test split folders.")
            for split_name, src_root in existing_splits.items():
                dst_root = os.path.join(CUB_SPLIT_ROOT, split_name)
                if not os.path.exists(dst_root):
                    shutil.copytree(src_root, dst_root)
            print("✅ CUB-200-2011 Data Structure Ready! (official split found)")
            return

        # الحالة 2: مجلد واحد فيه كل الأصناف (200 مجلد، زي CUB_200_2011/images/)
        # → نعمل تقسيم يدوي class-disjoint: 100/50/50
        images_root = _find_images_folder(dataset_path) or _find_class_dirs(dataset_path)
        if images_root is None:
            raise RuntimeError(
                "Could not locate CUB-200-2011 class directories in dataset."
            )

        print(f"📁 Found images root: {images_root}")
        classes = sorted([d for d in os.listdir(images_root)
                           if os.path.isdir(os.path.join(images_root, d))])
        print(f"📦 Found {len(classes)} classes.")

        rng = random.Random(42)  # تقسيم ثابت وقابل لإعادة الإنتاج بين الجلسات —
                                  # غيّر القيمة لو عايز تقسيم مختلف.
        rng.shuffle(classes)
        n_total = len(classes)
        # نسبة قياسية لأبحاث Few-Shot على CUB: 100 train / 50 val / 50 test
        # (لو عدد الأصناف الفعلي مختلف عن 200 بسبب اختلاف نسخة الداتاست، بنقسم
        # بنفس النسبة تقريبًا: 50% / 25% / 25%)
        if n_total == 200:
            n_train, n_val = 100, 50
        else:
            n_train = int(round(n_total * 0.5))
            n_val   = int(round(n_total * 0.25))
        splits = {
            "train": classes[:n_train],
            "val":   classes[n_train:n_train + n_val],
            "test":  classes[n_train + n_val:],
        }
        print(f"   Split sizes → train: {len(splits['train'])}, "
              f"val: {len(splits['val'])}, test: {len(splits['test'])}")

        for split_name, split_classes in splits.items():
            os.makedirs(os.path.join(CUB_SPLIT_ROOT, split_name), exist_ok=True)
            for c in split_classes:
                src = os.path.join(images_root, c)
                dst = os.path.join(CUB_SPLIT_ROOT, split_name, c)
                if not os.path.exists(dst):
                    shutil.copytree(src, dst)
        print("✅ CUB-200-2011 Data Structure Ready! (class-disjoint split)")
    except Exception as e:
        print(f"❌ Error preparing CUB-200-2011: {e}")
        raise


with StageTimer("Stage 0: Data Preparation (CUB-200-2011)"):
    prepare_cub200()

# ============================================
# 4. PROTOTYPE (Plain ProtoNet — no learnable norm, no gamma)
# ============================================

def compute_prototype(z_support):
    """Class prototype = arithmetic mean of raw support embeddings."""
    return z_support.mean(dim=0)

# ============================================
# 5. MODEL DEFINITION — MobileNetV3-LARGE
# ============================================

def get_mobilenet_finetuned(num_classes=100, supervised=True):
    print(f"🔄 Loading MobileNetV3-SMALL pretrained (num_classes={num_classes})...")
    weights = models.MobileNet_V3_Small_Weights.DEFAULT
    model   = models.mobilenet_v3_small(weights=weights)

    if supervised:
        in_features = model.classifier[3].in_features
        model.classifier[3] = nn.Linear(in_features, num_classes)
    else:
        model.classifier = nn.Identity()

    return model


def build_protonet_for_inference(checkpoint_path, num_classes=100):
    base_model = get_mobilenet_finetuned(num_classes=num_classes, supervised=True)
    base_model.classifier = nn.Identity()
    model = MobileProtoNet(base_model)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.to(device)
    model.eval()
    print(f"✅ Loaded checkpoint: {checkpoint_path}")
    return model


class MobileProtoNet(nn.Module):
    """
    Plain ProtoNet: prototype = mean of support embeddings,
    logits = -squared_euclidean_distance(query, prototype).
    لا يوجد LearnableNorm ولا gamma (بدون أي معاملات إضافية قابلة للتعلم
    خارج الـ backbone نفسه).
    """
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, support_x, query_x, n_way, k_shot):
        x = torch.cat([support_x, query_x], dim=0)
        x = self.backbone.features(x)
        x = self.backbone.avgpool(x)
        z = torch.flatten(x, 1)

        z_support = z[:support_x.size(0)]
        z_query   = z[support_x.size(0):]

        z_proto = torch.stack([
            compute_prototype(z_support[i * k_shot:(i + 1) * k_shot])
            for i in range(n_way)
        ])

        # ProtoNet القياسي يستخدم مربع المسافة الإقليدية
        dists = torch.cdist(z_query, z_proto) ** 2
        logits = -dists
        return logits

# ============================================
# 6. DATA LOADERS (parametrized by input_size)
# ============================================

def get_resize_dim(crop_dim):
    return round(crop_dim * 256 / 224)


def build_transforms(input_size):
    resize_dim = get_resize_dim(input_size)

    train_transform = transforms.Compose([
        transforms.Resize((resize_dim, resize_dim)),
        transforms.RandomCrop(input_size),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.4, 0.4, 0.4),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    test_transform = transforms.Compose([
        transforms.Resize((resize_dim, resize_dim)),
        transforms.CenterCrop(input_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return train_transform, test_transform



# ===== CONTROL: 84 → 224 (upsampled) =====
# نفس معلومات إدخال 84 بالظبط: resize 96 → crop 84 → bilinear upsample لـ 224.
# مفيش أي تفاصيل زيادة عن 84؛ الشبكة بتشوف مقياس 224 بس.
CTRL_NAME, CTRL_CROP, CTRL_SIZE = "84up224", 84, 224
CTRL_RESIZE = round(CTRL_CROP * 256 / 224)   # = 96، نفس خط 84 الأصلي بالظبط
_NORM = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

def build_transforms_ctrl():
    up = transforms.Resize((CTRL_SIZE, CTRL_SIZE),
                           interpolation=transforms.InterpolationMode.BILINEAR)
    train_t = transforms.Compose([
        transforms.Resize((CTRL_RESIZE, CTRL_RESIZE)),
        transforms.RandomCrop(CTRL_CROP),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.4, 0.4, 0.4),
        up, transforms.ToTensor(), _NORM,
    ])
    test_t = transforms.Compose([
        transforms.Resize((CTRL_RESIZE, CTRL_RESIZE)),
        transforms.CenterCrop(CTRL_CROP),
        up, transforms.ToTensor(), _NORM,
    ])
    return train_t, test_t

CUB_ROOT = CUB_SPLIT_ROOT

# batch size أقل عند الدقات الكبيرة عشان نتجنب overflow في VRAM. القيم دي
# مقلّلة عن نسخة Small (تقريبًا للنصف) لأن MobileNetV3-Large backbone أتقل
# في الذاكرة (~5.4M params وFLOPs أعلى بشكل ملحوظ من Small). القيم تقديرية
# — لو حصل CUDA Out Of Memory قلّلها أكتر حسب الـ GPU المتاح فعليًا.
BATCH_SIZE_BY_RESOLUTION = {84: 64, 112: 48, 168: 32, 224: 32, 256: 16}


class EpisodicSampler:
    def __init__(self, root, transform):
        self.root      = root
        self.transform = transform
        self.classes   = sorted(os.listdir(root))
        self.data      = {}
        for c in self.classes:
            cls_path = os.path.join(root, c)
            self.data[c] = [
                os.path.join(cls_path, img)
                for img in os.listdir(cls_path)
                if img.lower().endswith(('jpg', 'jpeg', 'png'))
            ]

    def get_episode(self, n_way, k_shot, q_query):
        selected_classes = random.sample(self.classes, n_way)
        support_imgs, query_imgs, query_labels = [], [], []

        for i, cls in enumerate(selected_classes):
            available_imgs = self.data[cls]
            needed = k_shot + q_query
            if len(available_imgs) < needed:
                sampled_paths = random.choices(available_imgs, k=needed)
            else:
                sampled_paths = random.sample(available_imgs, needed)

            for path in sampled_paths[:k_shot]:
                img = Image.open(path).convert('RGB')
                support_imgs.append(self.transform(img))

            for path in sampled_paths[k_shot:]:
                img = Image.open(path).convert('RGB')
                query_imgs.append(self.transform(img))
                query_labels.append(i)

        return (torch.stack(support_imgs),
                torch.stack(query_imgs),
                torch.tensor(query_labels))

# ============================================
# 6b. MATCHED-EPISODE EVALUATION (fixed episode set)
# ============================================

def generate_fixed_episodes(root, n_way, k_shot, q_query, n_episodes, seed):
    rng = random.Random(seed)  # RNG محلي مستقل — لا يمس حالة random العامة
    classes = sorted(os.listdir(root))
    data = {}
    for c in classes:
        cls_path = os.path.join(root, c)
        data[c] = [
            os.path.join(cls_path, img)
            for img in os.listdir(cls_path)
            if img.lower().endswith(('jpg', 'jpeg', 'png'))
        ]

    episodes = []
    for _ in range(n_episodes):
        selected_classes = rng.sample(classes, n_way)
        support_paths, query_paths, query_labels = [], [], []

        for i, cls in enumerate(selected_classes):
            available_imgs = data[cls]
            needed = k_shot + q_query
            if len(available_imgs) < needed:
                sampled_paths = rng.choices(available_imgs, k=needed)
            else:
                sampled_paths = rng.sample(available_imgs, needed)

            support_paths.extend(sampled_paths[:k_shot])
            query_paths.extend(sampled_paths[k_shot:])
            query_labels.extend([i] * q_query)

        episodes.append({
            "classes": selected_classes,
            "support_paths": support_paths,
            "query_paths": query_paths,
            "query_labels": query_labels,
        })

    return episodes


def _load_episode_tensors(episode_def, transform, n_way, k_shot):
    support_imgs = [transform(Image.open(p).convert('RGB'))
                     for p in episode_def["support_paths"]]
    query_imgs   = [transform(Image.open(p).convert('RGB'))
                     for p in episode_def["query_paths"]]
    query_labels = torch.tensor(episode_def["query_labels"])
    return torch.stack(support_imgs), torch.stack(query_imgs), query_labels


@torch.no_grad()
def evaluate_fixed_episodes(model, episode_defs, transform, n_way, k_shot, desc=""):
    """يقيّم موديل على مجموعة episodes ثابتة (paths محددة سلفًا) ويرجّع
    مصفوفة دقة لكل episode على حدة (بدون تجميع)، عشان تُستخدم في اختبار
    paired لاحقًا."""
    model.eval()
    accs = []
    for ep_def in tqdm(episode_defs, desc=f"Matched-Eval {desc}", leave=False):
        sx, qx, qy = _load_episode_tensors(ep_def, transform, n_way, k_shot)
        sx, qx, qy = sx.to(device), qx.to(device), qy.to(device)
        logits = model(sx, qx, n_way, k_shot)
        preds  = logits.argmax(dim=1)
        accs.append((preds == qy).float().mean().item())
    return np.array(accs)


def paired_resolution_test(acc_84, acc_224, label=""):
    """اختبار paired بين نفس الـ episodes بدقتين مختلفتين.
    Δ_i = Acc224_i - Acc84_i ، H0: E[Δ] = 0."""
    assert len(acc_84) == len(acc_224), "عدد الـ episodes لازم يتطابق للمقارنة الـ paired"
    delta = acc_224 - acc_84
    mean_delta = delta.mean()
    std_delta  = delta.std(ddof=1)
    n = len(delta)
    se = std_delta / np.sqrt(n)
    ci95 = 1.96 * se

    t_stat, t_pval = scipy_stats.ttest_rel(acc_224, acc_84)
    try:
        w_stat, w_pval = scipy_stats.wilcoxon(acc_224, acc_84)
    except ValueError:
        # Wilcoxon بيفشل لو كل الفروق = صفر
        w_stat, w_pval = float('nan'), float('nan')

    print(f"\n📈 Paired 84x84 vs 224x224 — {label}")
    print(f"   n episodes        : {n}")
    print(f"   Mean Acc @84       : {acc_84.mean():.2%}")
    print(f"   Mean Acc @224      : {acc_224.mean():.2%}")
    print(f"   Mean Δ (224-84)    : {mean_delta:+.2%}  (95% CI ±{ci95:.2%})")
    print(f"   Paired t-test      : t={t_stat:.4f}, p={t_pval:.6f}")
    print(f"   Wilcoxon signed-rank: W={w_stat:.4f}, p={w_pval:.6f}")

    return {
        "n": n,
        "mean_delta": mean_delta,
        "ci95_delta": ci95,
        "t_stat": t_stat,
        "t_pval": t_pval,
        "w_stat": w_stat,
        "w_pval": w_pval,
        "acc_84_mean": acc_84.mean(),
        "acc_224_mean": acc_224.mean(),
    }

# ============================================
# 7. EXECUTION STAGES
# ============================================

def stage1_finetuning(train_loader_simple, epochs=10, num_classes=100):
    print(f"\n🦾 STAGE 1: Fine-tuning MobileNet on CUB-200-2011 for {epochs} epochs...")

    model     = get_mobilenet_finetuned(num_classes=num_classes, supervised=True).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9,
                                 nesterov=True, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0

        pbar = tqdm(train_loader_simple, desc=f"Fine-tune Ep {epoch+1}", leave=False)
        for imgs, labels in pbar:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            preds    = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
            pbar.set_postfix({'Acc': f"{correct/total:.1%}"})

        scheduler.step()
        print(f"   Ep {epoch+1}: Loss {total_loss/len(train_loader_simple):.4f} | "
              f"Train Acc {correct/total:.2%}")

    print("✅ Fine-tuning Complete.")
    return model


def stage2_metatrain(pretrained_state_dict, train_sampler, val_sampler,
                      n_way=5, k_shot=1, episodes=2000, num_classes=100,
                      ckpt_path="best_mobile_proto_cub200.pth"):
    print(f"\n🧠 STAGE 2: Meta-Training (ProtoNet) on CUB-200-2011 "
          f"{n_way}-way {k_shot}-shot...")

    base_model = get_mobilenet_finetuned(num_classes=num_classes, supervised=True)
    base_model.load_state_dict(pretrained_state_dict)
    base_model.classifier = nn.Identity()

    model = MobileProtoNet(base_model).to(device)

    # لا يوجد learnable_norm أو gamma الآن، لذلك مجموعة معاملات واحدة فقط للـ backbone
    optimizer = torch.optim.Adam(model.backbone.parameters(), lr=1e-5)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    patience     = 0

    for ep in tqdm(range(episodes), desc="Meta-Train"):
        sx, qx, qy = train_sampler.get_episode(n_way, k_shot, q_query=15)
        sx, qx, qy = sx.to(device), qx.to(device), qy.to(device)

        model.train()
        optimizer.zero_grad()
        logits = model(sx, qx, n_way, k_shot)
        loss   = criterion(logits, qy)
        loss.backward()
        optimizer.step()

        if (ep + 1) % 200 == 0:
            val_acc, _ = evaluate(model, val_sampler, n_way, k_shot,
                                   episodes=200, silent=True)
            print(f"\n   Ep {ep+1}: Loss {loss.item():.4f} | Val Acc: {val_acc:.2%}")

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(model.state_dict(), ckpt_path)
                patience = 0
            else:
                patience += 1
                if patience > 10:
                    print("⏹️ Early stopping triggered.")
                    break

    return model


@torch.no_grad()
def evaluate(model, sampler, n_way, k_shot, episodes=500, silent=False):
    model.eval()
    accs     = []
    iterator = range(episodes) if silent else tqdm(range(episodes), desc="Evaluating")

    for _ in iterator:
        sx, qx, qy = sampler.get_episode(n_way, k_shot, q_query=15)
        sx, qx, qy = sx.to(device), qx.to(device), qy.to(device)

        logits = model(sx, qx, n_way, k_shot)
        preds  = logits.argmax(dim=1)
        accs.append((preds == qy).float().mean().item())

    mean_acc = np.mean(accs)
    ci95     = 1.96 * np.std(accs) / np.sqrt(episodes)

    if not silent:
        print(f"📊 Result: {mean_acc:.2%} ± {ci95:.2%}")

    return mean_acc, ci95

# ============================================
# 8. RUN ONE FULL EXPERIMENT (given input_size + seed)
# ============================================

NUM_TRAIN_CLASSES = 100  # CUB-200-2011 few-shot train split size (100/50/50)

def run_experiment(input_size, seed, epochs=10, episodes=2000):
    print("\n" + "#"*60)
    print(f"### RUNNING EXPERIMENT: input_size={input_size}x{input_size} | seed={seed}")
    print("#"*60)

    set_seed(seed)

    train_transform, test_transform = build_transforms_ctrl()   # CONTROL 84→224

    train_dataset_simple = datasets.ImageFolder(
        root=os.path.join(CUB_ROOT, "train"),
        transform=train_transform
    )
    batch_size = BATCH_SIZE_BY_RESOLUTION.get(CTRL_SIZE, 32)
    train_loader_simple = DataLoader(
        train_dataset_simple,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )
    print(f"   Batch size: {batch_size}")

    train_sampler = EpisodicSampler(os.path.join(CUB_ROOT, "train"), test_transform)
    val_sampler   = EpisodicSampler(os.path.join(CUB_ROOT, "val"),   test_transform)
    test_sampler  = EpisodicSampler(os.path.join(CUB_ROOT, "test"),  test_transform)

    tag = f"{CTRL_NAME}_seed{seed}"

    # ── Stage 1 ──────────────────────────────
    backbone_ckpt = os.path.join(WORK_ROOT, f"backbone_pretrained_cub200_{tag}.pth")
    if not os.path.exists(backbone_ckpt):
        with StageTimer(f"Stage 1: Fine-Tuning ({tag})"):
            ft_model = stage1_finetuning(train_loader_simple, epochs=epochs,
                                          num_classes=NUM_TRAIN_CLASSES)
            torch.save(ft_model.state_dict(), backbone_ckpt)
    else:
        print(f"✅ Pre-trained backbone found for {tag}. Skipping Stage 1.")
    pretrained_state_dict = torch.load(backbone_ckpt, map_location=device)

    results = {}

    for k_shot in [1, 5]:
        print("\n" + "="*40)
        print(f"🎯 TASK: 5-way {k_shot}-shot ({tag})")
        print("="*40)

        ckpt_path = os.path.join(
            WORK_ROOT, f"best_mobile_proto_cub200_{tag}_{k_shot}shot.pth"
        )

        with StageTimer(f"Stage 2: Meta-Training {k_shot}-shot ({tag})"):
            stage2_metatrain(pretrained_state_dict, train_sampler, val_sampler,
                              n_way=5, k_shot=k_shot, episodes=episodes,
                              num_classes=NUM_TRAIN_CLASSES, ckpt_path=ckpt_path)

        with StageTimer(f"Stage 3: Test Evaluation {k_shot}-shot ({tag})"):
            best_model = build_protonet_for_inference(
                checkpoint_path=ckpt_path, num_classes=NUM_TRAIN_CLASSES
            )
            mean_acc, ci95 = evaluate(best_model, test_sampler, n_way=5, k_shot=k_shot)
            results[k_shot] = {"mean_acc": mean_acc, "ci95": ci95, "ckpt_path": ckpt_path}

        if k_shot == 5:
            print("\n" + "="*40)
            print(f"📐 MODEL SIZE & COMPUTE REPORT ({tag})")
            print("="*40)
            report_full_model_stats(best_model, input_size=input_size)

    return results

# ============================================
# 8b. RESUME / PROGRESS TRACKING
# ============================================
# ملف تقدّم بيتحدّث بعد كل (input_size, seed, k_shot) يخلص — لو التجربة
# اتقطعت (انقطع النت، قفلت الجلسة، ...) وشغّلت الكود تاني (بعد ما تضيف
# الـ Output بتاع النسخة المحفوظة كـ Input زي ما اتشرح فوق)، هيقرأ الملف
# ده ويكمّل من عند آخر حاجة خلصها، من غير ما يعيد أي تدريب أو تقييم اتعمل
# قبل كده. الحفظ atomic (كتابة لملف مؤقت ثم rename) عشان لو الجلسة اتقفلت
# في نص الكتابة، الملف الأصلي يفضل سليم.

PROGRESS_FILE = os.path.join(WORK_ROOT, "progress.json")


def load_progress():
    """يرجّع (all_results, matched_accs) من ملف التقدّم لو موجود، وإلا
    قواميس فاضية. all_results بنفس شكل المتغيّر all_results في MAIN.
    matched_accs بنفس شكل matched_accs (مصفوفات per-episode لكل
    k_shot/seed/input_size، مستخدمة في الـ Matched-Episode Evaluation)."""
    if not os.path.exists(PROGRESS_FILE):
        return {}, {}
    try:
        with open(PROGRESS_FILE, "r") as f:
            data = json.load(f)
    except (json.JSONDecodeError, OSError) as e:
        print(f"⚠️  ملف التقدّم {PROGRESS_FILE} تالف ({e}) — هنبدأ من الصفر.")
        return {}, {}

    all_results = {}
    for res_str, seeds in data.get("results", {}).items():
        res = int(res_str)
        all_results[res] = {}
        for seed_str, kshots in seeds.items():
            seed = int(seed_str)
            all_results[res][seed] = {}
            for k_str, v in kshots.items():
                all_results[res][seed][int(k_str)] = {
                    "mean_acc": v["mean_acc"],
                    "ci95": v["ci95"],
                    "ckpt_path": v["ckpt_path"],
                }

    matched_accs = {}
    for k_str, seeds in data.get("matched", {}).items():
        k_shot = int(k_str)
        matched_accs[k_shot] = {}
        for seed_str, res_dict in seeds.items():
            seed = int(seed_str)
            matched_accs[k_shot][seed] = {
                84:  np.array(res_dict["84"]),
                224: np.array(res_dict["224"]),
            }

    if all_results or matched_accs:
        print(f"\n♻️  لقينا تقدّم سابق في {PROGRESS_FILE}:")
        for res, seeds in all_results.items():
            print(f"    {res}px: seeds خلّصت = {sorted(seeds.keys())}")
        for k_shot, seeds in matched_accs.items():
            print(f"    Matched-Eval {k_shot}-shot: seeds خلّصت = {sorted(seeds.keys())}")

    return all_results, matched_accs


def save_progress(all_results, matched_accs):
    """يحفظ التقدّم الحالي بطريقة atomic (كتابة لملف مؤقت ثم rename)."""
    report = {
        "results": {
            str(res): {
                str(seed): {
                    str(k_shot): {
                        "mean_acc": float(v["mean_acc"]),
                        "ci95": float(v["ci95"]),
                        "ckpt_path": v["ckpt_path"],
                    }
                    for k_shot, v in kshots.items()
                }
                for seed, kshots in seeds.items()
            }
            for res, seeds in all_results.items()
        },
        "matched": {
            str(k_shot): {
                str(seed): {
                    "84":  res_dict[84].tolist(),
                    "224": res_dict[224].tolist(),
                }
                for seed, res_dict in seeds.items()
            }
            for k_shot, seeds in matched_accs.items()
        },
    }
    tmp_path = PROGRESS_FILE + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(report, f, indent=2)
    os.replace(tmp_path, PROGRESS_FILE)  # rename ذرّي — إما ينجح كامل أو مايحصلش خالص

    # 🆕 طباعة تذكير بعد كل حفظ — عشان تفتكر تعمل commit بانتظام
    print(f"💾 اتحفظ التقدّم محليًا في {PROGRESS_FILE} — "
          f"متنساش تعمل 'Save Version' من وقت للتاني عشان التقدّم يتحفظ فعليًا.")

# ============================================
# 8c. AUTO-PUSH TO GITHUB (بديل تلقائي للتنزيل اليدوي)
# ============================================
# بعد كل seed، بنعمل push لكل محتوى WORK_ROOT (progress.json + كل الـ
# checkpoints) لمستودع GitHub بتاعك. لو فشل لأي سبب، بنطبع تحذير ونكمّل
# التدريب عادي — الفشل هنا مايوقفش التجربة خالص.

_github_push_warned_once = False

# ملحوظة: _run_git_cmd معرّفة فوق في قسم 0 (PATHS) عشان تُستخدم في
# الاسترجاع التلقائي من GitHub قبل بداية التدريب، وبنعيد استخدامها هنا.


def push_results_to_github(commit_message="Update results"):
    """🆕 يعمل push تلقائي لملفات WORK_ROOT (progress.json + checkpoints)
    لمستودع GitHub محدد في GITHUB_REPO. أي فشل هنا بيتحول لتحذير فقط —
    مايوقفش التدريب خالص."""
    global _github_push_warned_once

    if GITHUB_REPO.startswith("your-username/"):
        if not _github_push_warned_once:
            print("ℹ️  GITHUB_REPO لسه مش معدّل (لسه على القيمة الافتراضية) — "
                  "هنتجاهل GitHub Auto-Push. غيّر GITHUB_REPO في بداية الكود "
                  "لو عايز تفعّل الخاصية دي.")
            _github_push_warned_once = True
        return False

    try:
        import subprocess as _sp  # noqa: F401 (تأكيد إنه متاح هنا)
    except ImportError:
        pass

    token = _get_github_token()
    if not token:
        print("⚠️  مقدرتش أجيب GITHUB_TOKEN من Colab Secrets. ضيفه من أيقونة المفتاح "
              "باسم GITHUB_TOKEN وفعّل Notebook access. هنتجاهل GitHub Auto-Push المرة دي.")
        return False

    remote_url = f"https://{token}@github.com/{GITHUB_REPO}.git"

    try:
        # Clone لو أول مرة، وإلا pull عادي
        if not os.path.exists(os.path.join(GITHUB_CLONE_DIR, ".git")):
            print(f"\n📥 بنعمل clone لأول مرة لمستودع النتائج ({GITHUB_REPO})...")
            ok, out = _run_git_cmd([
                "clone", "--depth", "1", "--branch", GITHUB_BRANCH,
                remote_url, GITHUB_CLONE_DIR
            ])
            if not ok:
                # ممكن يكون البranch مش موجود لسه (مستودع فاضي) → نعمل init بدالها
                print("ℹ️  الـ clone فشل (غالبًا المستودع فاضي لسه) — هنعمل init محلي بدل كده.")
                os.makedirs(GITHUB_CLONE_DIR, exist_ok=True)
                _run_git_cmd(["init"], cwd=GITHUB_CLONE_DIR)
                _run_git_cmd(["checkout", "-b", GITHUB_BRANCH], cwd=GITHUB_CLONE_DIR)
                _run_git_cmd(["remote", "add", "origin", remote_url], cwd=GITHUB_CLONE_DIR)
        else:
            ok, out = _run_git_cmd(["pull", "--rebase", "origin", GITHUB_BRANCH],
                                    cwd=GITHUB_CLONE_DIR)
            if not ok:
                print(f"⚠️  git pull فشل (هنكمّل ونحاول الـ push عادي):\n{out}")

        _run_git_cmd(["config", "user.email", "kaggle-bot@example.com"], cwd=GITHUB_CLONE_DIR)
        _run_git_cmd(["config", "user.name", "Kaggle Auto-Push"], cwd=GITHUB_CLONE_DIR)

        # ننسخ كل محتوى WORK_ROOT لمجلد فرعي جوه المستودع
        dst_dir = os.path.join(GITHUB_CLONE_DIR, GITHUB_SUBFOLDER)
        os.makedirs(dst_dir, exist_ok=True)
        for item in os.listdir(WORK_ROOT):
            src_path = os.path.join(WORK_ROOT, item)
            dst_path = os.path.join(dst_dir, item)
            if os.path.isfile(src_path):
                shutil.copy2(src_path, dst_path)

        ok, out = _run_git_cmd(["add", "-A"], cwd=GITHUB_CLONE_DIR)
        ok, out = _run_git_cmd(["commit", "-m", commit_message], cwd=GITHUB_CLONE_DIR)
        if not ok and "nothing to commit" not in out.lower():
            print(f"⚠️  git commit فشل:\n{out}")
            return False

        ok, out = _run_git_cmd(
            ["push", "-u", "origin", f"HEAD:{GITHUB_BRANCH}"],
            cwd=GITHUB_CLONE_DIR,
            env={**os.environ, "GIT_TERMINAL_PROMPT": "0"}
        )
        if not ok:
            print(f"⚠️  git push فشل (هنكمّل التدريب عادي، الملفات لسه محفوظة "
                  f"محليًا في /content):\n{out}")
            return False

        print(f"🚀 اتعمل push ناجح للمستودع {GITHUB_REPO} (فرع {GITHUB_BRANCH}) — "
              f"الملفات دلوقتي متاحة على GitHub.")
        return True

    except Exception as e:
        print(f"⚠️  GitHub Auto-Push فشل بشكل غير متوقع ({e}) — هنكمّل التدريب عادي.")
        return False



# ============================================
# 9. MAIN — CONTROL: 84 → 224 upsampled (3-way comparison)
# ============================================
# بيشغّل حالة الـ control بس (نفس seeds، نفس الـ 2000 matched episodes)، ويقارنها
# بنتايج 84 و224 الأصلية اللي لازم تكون اتسجّلت قبل كده في نفس الـ WORK_ROOT
# (progress.json بتتسترجع تلقائيًا من GitHub زي ما بتحصل في التجارب الأساسية).
# نتايج الـ control بتتحفظ في ملف منفصل progress_control.json (مبيلمسش progress.json).

CTRL_PROGRESS = os.path.join(WORK_ROOT, "progress_control.json")

def load_ctrl_progress():
    if not os.path.exists(CTRL_PROGRESS):
        return {"results": {}, "matched": {}}
    try:
        with open(CTRL_PROGRESS) as f:
            return json.load(f)
    except Exception as e:
        print(f"⚠️  {CTRL_PROGRESS} تالف ({e}) — هنبدأ الـ control من الصفر.")
        return {"results": {}, "matched": {}}

def save_ctrl_progress(cp):
    tmp = CTRL_PROGRESS + ".tmp"
    with open(tmp, "w") as f:
        json.dump(cp, f, indent=2)
    os.replace(tmp, CTRL_PROGRESS)
    print(f"💾 اتحفظ {CTRL_PROGRESS}")

def paired_compare(a, b, name_a, name_b, label):
    """Δ = b − a على نفس الـ episodes."""
    d = b - a
    n = len(d)
    ci = 1.96 * d.std(ddof=1) / np.sqrt(n)
    t, tp = scipy_stats.ttest_rel(b, a)
    try:
        w, wp = scipy_stats.wilcoxon(b, a)
    except ValueError:
        w, wp = float("nan"), float("nan")
    print(f"   {name_b:>8s} − {name_a:<8s}: Δ={d.mean():+.2%} (±{ci:.2%}) | "
          f"t-test p={tp:.2e} | Wilcoxon p={wp:.2e}   [{label}]")
    return d.mean()


if __name__ == "__main__":
    SEEDS   = [1]   # seed واحد الأول؛ زوّد [1, 2, 3] بعد كده (بيكمّل من progress_control.json)
    N_WAY   = 5
    Q_QUERY = 15
    N_MATCHED_EPISODES = 2000
    MATCHED_EPISODES_SEED = 999   # لازم يفضل نفس القيمة في التجارب الأساسية

    base_results, base_matched = load_progress()
    cp = load_ctrl_progress()

    # ── تدريب + تقييم الـ control لكل seed ──
    for seed in SEEDS:
        s = str(seed)
        if s in cp["results"] and "1" in cp["results"][s] and "5" in cp["results"][s]:
            print(f"✅ control seed={seed} موجود — تخطّي التدريب.")
            continue
        res = run_experiment(224, seed, epochs=10, episodes=2000)
        cp["results"][s] = {str(k): {"mean_acc": float(v["mean_acc"]), "ci95": float(v["ci95"]),
                                      "ckpt_path": v["ckpt_path"]} for k, v in res.items()}
        save_ctrl_progress(cp)
        push_results_to_github(commit_message=f"Control 84up224: seed={seed} trained")

    # ── Matched-episode eval للـ control على نفس الـ episodes الثابتة ──
    with StageTimer("Control: Matched-Episode Generation"):
        fixed_episodes = {
            k: generate_fixed_episodes(root=os.path.join(CUB_ROOT, "test"), n_way=N_WAY, k_shot=k,
                                        q_query=Q_QUERY, n_episodes=N_MATCHED_EPISODES,
                                        seed=MATCHED_EPISODES_SEED)
            for k in [1, 5]
        }
    _, test_t_ctrl = build_transforms_ctrl()

    for k in [1, 5]:
        cp["matched"].setdefault(str(k), {})
        for seed in SEEDS:
            s = str(seed)
            if s in cp["matched"][str(k)]:
                continue
            ckpt = cp["results"][s][str(k)]["ckpt_path"]
            if not os.path.exists(ckpt):  # لو المسار اتغيّر بعد restore
                ckpt = os.path.join(WORK_ROOT, os.path.basename(ckpt))
            model = build_protonet_for_inference(ckpt, num_classes=NUM_TRAIN_CLASSES)
            accs = evaluate_fixed_episodes(model, fixed_episodes[k], test_t_ctrl, N_WAY, k,
                                            desc=f"84up224 seed{seed} {k}shot")
            cp["matched"][str(k)][s] = accs.tolist()
            save_ctrl_progress(cp)
            push_results_to_github(commit_message=f"Control 84up224: matched {k}-shot seed={seed}")

    # ── التحليل: 84 vs 84up224 vs 224 ──
    print("\n" + "=" * 70)
    print("🧪 CONTROL ANALYSIS: 84 (original) vs 84→224 (upsampled) vs 224 (original)")
    print("=" * 70)
    for k in [1, 5]:
        usable = [sd for sd in SEEDS
                  if sd in base_matched.get(k, {}) and str(sd) in cp["matched"][str(k)]]
        if not usable:
            print(f"\n⚠️  {k}-shot: مفيش matched arrays للـ 84/224 الأصلية في progress.json — "
                  f"شغّل التجارب الأساسية الأول (أو اتأكد من الاسترجاع من GitHub).")
            continue
        A84  = {sd: base_matched[k][sd][84]  for sd in usable}
        A224 = {sd: base_matched[k][sd][224] for sd in usable}
        ACT  = {sd: np.array(cp["matched"][str(k)][str(sd)]) for sd in usable}
        assert all(len(ACT[sd]) == len(A84[sd]) == len(A224[sd]) == N_MATCHED_EPISODES
                   for sd in usable), "عدد الـ episodes مش متطابق"

        print(f"\n── {k}-shot ── (seeds: {usable})")
        for name, D in [("84 original", A84), ("84→224 up", ACT), ("224 original", A224)]:
            per_seed = [D[sd].mean() for sd in usable]
            print(f"   {name:13s}: {np.mean(per_seed):.2%} ± {np.std(per_seed):.2%}   "
                  f"(per seed: {[f'{a:.2%}' for a in per_seed]})")

        p84, pct, p224 = (np.concatenate([D[sd] for sd in usable]) for D in (A84, ACT, A224))
        print(f"   Paired tests (pooled {len(usable)} seeds × {N_MATCHED_EPISODES} episodes):")
        total  = paired_compare(p84, p224, "84", "224", "total effect")
        scale  = paired_compare(p84, pct,  "84", "84up224", "(b) scale/format only, no new detail")
        detail = paired_compare(pct, p224, "84up224", "224", "(a) extra visual detail")
        print(f"   Decomposition: total {total:+.2%} = scale/pretrain-match {scale:+.2%} "
              f"+ extra detail {detail:+.2%}")

    print("\n📖 التفسير:")
    print("  • 84up224 ≈ 224  والفرق بينهم غير معنوي → المكسب مصدره مقياس 224 (تفسير ب).")
    print("  • 84up224 ≈ 84   والفرق مع 224 كبير       → المكسب مصدره التفاصيل البصرية (تفسير أ).")
    print("  • بينهم → الاتنين مساهمين؛ استخدم الـ decomposition فوق بالأرقام.")
    print_stage_time_summary()

📦 Installing thop...


In [ ]:
import os
import json
import subprocess
try:
    import kagglehub
except ImportError:
    print(" Installing kagglehub...")
    os.system('pip install -q kagglehub')
    import kagglehub

try:
    from thop import profile as thop_profile
except ImportError:
    print(" Installing thop...")
    os.system('pip install -q thop')
    from thop import profile as thop_profile

try:
    from scipy import stats as scipy_stats
except ImportError:
    print(" Installing scipy...")
    os.system('pip install -q scipy')
    from scipy import stats as scipy_stats

import time
import random
import shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from tqdm import tqdm
from PIL import Image


CUB_KAGGLEHUB_ID = "wenewone/cub2002011"

WORK_ROOT      = os.path.join(os.getcwd(), "cub200_workdir")
CUB_SPLIT_ROOT = os.path.join(WORK_ROOT, "cub200_split")
os.makedirs(WORK_ROOT, exist_ok=True)

# ============================================================================
# 🆕 GITHUB AUTO-PUSH / AUTO-RESTORE
# ----------------------------------------------------------------------------
# الإعداد (مرة واحدة):
#  1) اعمل مستودع Private جديد على GitHub للنتائج بس.
#  2) اعمل Personal Access Token (classic) بصلاحية "repo".
#  3) خزّن الـ token باسم GITHUB_TOKEN:
#       - Colab : أيقونة المفتاح 🔑 (Secrets) على الشمال → Add new secret →
#                 الاسم GITHUB_TOKEN → فعّل "Notebook access".
#       - Kaggle: Add-ons → Secrets → GITHUB_TOKEN.
#       - أو متغير بيئة GITHUB_TOKEN.
#  4) غيّر GITHUB_REPO تحت لاسم مستودعك (username/reponame).
# لو أي حاجة فشلت، الكود بيطبع تحذير ويكمّل التدريب عادي.
# ============================================================================
GITHUB_REPO       = "Ahmed-Hamam/control84up224"  # ⚠️ غيّرها لمستودعك!
GITHUB_BRANCH     = "main"
GITHUB_CLONE_DIR  = os.path.join(os.getcwd(), "_github_results_repo")
GITHUB_SUBFOLDER  = "cub-Large_results"  # المجلد جوه المستودع

_github_push_warned_once = False


def _get_github_token():
    """يجيب الـ token من Colab Secrets أو Kaggle Secrets أو متغير بيئة."""
    try:
        from google.colab import userdata
        tok = userdata.get("GITHUB_TOKEN")
        if tok:
            return tok
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
        if tok:
            return tok
    except Exception:
        pass
    return os.environ.get("GITHUB_TOKEN")


def _run_git_cmd(args, cwd=None, env=None, secret=None):
    """ينفّذ أمر git ويرجّع (success, output). بيشيل الـ token من الـ output
    عشان ميظهرش في اللوجات."""
    result = subprocess.run(["git"] + args, cwd=cwd, env=env,
                            capture_output=True, text=True)
    ok = (result.returncode == 0)
    output = (result.stdout or "") + (result.stderr or "")
    if secret:
        output = output.replace(secret, "***")
    return ok, output


def _github_configured():
    return not GITHUB_REPO.startswith("your-username/")


def _restore_previous_workdir_from_github():
    """لو مفيش progress.json محلي، بيسحب آخر progress.json + checkpoints
    من GitHub ويكمّل من عندهم. بيعمل clone نضيف كل مرة، وبيطبع السبب
    الحقيقي لو فشل (مش رسالة عامة)."""
    if os.path.exists(os.path.join(WORK_ROOT, "progress.json")):
        print("✅ progress.json موجود بالفعل في WORK_ROOT — مفيش استرجاع.")
        return
    if not _github_configured():
        print("ℹ️  GITHUB_REPO لسه مش معدّل — هنتجاهل الاسترجاع من GitHub.")
        return
    token = _get_github_token()
    if not token:
        print("⚠️  مفيش GITHUB_TOKEN (تأكد إن Secret اسمه GITHUB_TOKEN و"
              "\"Notebook access\" مفعّل للـ notebook ده) — هنبدأ من الصفر.")
        return

    print(f"\n🔍 بنحاول نسترجع نسخة سابقة من GitHub ({GITHUB_REPO}, branch={GITHUB_BRANCH})...")
    remote_url = f"https://{token}@github.com/{GITHUB_REPO}.git"
    try:
        # clone نضيف دايمًا (يمسح أي clone قديم أو فاسد من محاولة سابقة)
        if os.path.exists(GITHUB_CLONE_DIR):
            shutil.rmtree(GITHUB_CLONE_DIR, ignore_errors=True)

        ok, out = _run_git_cmd(
            ["clone", "--depth", "1", "--branch", GITHUB_BRANCH,
             remote_url, GITHUB_CLONE_DIR], secret=token)
        if not ok:
            print(f"⚠️  clone بالـ branch '{GITHUB_BRANCH}' فشل. السبب الحقيقي:\n{out}")
            print("   بنجرّب clone بالـ branch الافتراضي...")
            shutil.rmtree(GITHUB_CLONE_DIR, ignore_errors=True)
            ok, out = _run_git_cmd(
                ["clone", "--depth", "1", remote_url, GITHUB_CLONE_DIR], secret=token)
            if not ok:
                print(f"❌ الـ clone فشل تاني. السبب:\n{out}\n"
                      f"   (تأكد من اسم المستودع {GITHUB_REPO} وإن الـ token عنده صلاحية repo)")
                return

        src_dir = os.path.join(GITHUB_CLONE_DIR, GITHUB_SUBFOLDER)
        if not os.path.exists(os.path.join(src_dir, "progress.json")):
            top = sorted(d for d in os.listdir(GITHUB_CLONE_DIR) if d != ".git")
            with_prog = [d for d in top
                         if os.path.exists(os.path.join(GITHUB_CLONE_DIR, d, "progress.json"))]
            print(f"ℹ️  المستودع اتعمله clone بنجاح، بس مفيش progress.json في "
                  f"المجلد المتوقع '{GITHUB_SUBFOLDER}'.")
            print(f"   محتوى المستودع: {top}")
            print(f"   المجلدات اللي فيها progress.json: {with_prog}")
            if with_prog:
                print(f"   👉 لو التجربة دي موجودة في مجلد تاني، غيّر GITHUB_SUBFOLDER "
                      f"لاسمه (مثلًا '{with_prog[0]}').")
            return

        print(f"♻️  لقينا نسخة سابقة في {src_dir} — بننسخها لـ WORK_ROOT...")
        n = 0
        for item in os.listdir(src_dir):
            s = os.path.join(src_dir, item)
            d = os.path.join(WORK_ROOT, item)
            if os.path.isfile(s) and not os.path.exists(d):
                shutil.copy2(s, d)
                n += 1
        print(f"✅ تم استرجاع {n} ملف — الكود هيكمّل من آخر seed اتسجّل.")
    except Exception as e:
        print(f"⚠️  الاسترجاع من GitHub فشل ({e}) — هنبدأ من الصفر.")


def push_results_to_github(commit_message="Update results"):
    """يعمل push لملفات WORK_ROOT (progress.json + checkpoints، الملفات
    فقط من غير مجلد الداتا) لمستودع GitHub. أي فشل = تحذير فقط."""
    global _github_push_warned_once

    if not _github_configured():
        if not _github_push_warned_once:
            print("ℹ️  GITHUB_REPO لسه مش معدّل — GitHub Auto-Push متعطّل.")
            _github_push_warned_once = True
        return False

    token = _get_github_token()
    if not token:
        print("⚠️  مقدرتش أجيب GITHUB_TOKEN — هنتجاهل الـ push المرة دي.")
        return False

    remote_url = f"https://{token}@github.com/{GITHUB_REPO}.git"
    try:
        if not os.path.exists(os.path.join(GITHUB_CLONE_DIR, ".git")):
            print(f"\n📥 clone لأول مرة لمستودع النتائج ({GITHUB_REPO})...")
            ok, out = _run_git_cmd(
                ["clone", "--depth", "1", "--branch", GITHUB_BRANCH,
                 remote_url, GITHUB_CLONE_DIR], secret=token)
            if not ok:
                print("ℹ️  الـ clone فشل (غالبًا المستودع فاضي) — هنعمل init محلي.")
                os.makedirs(GITHUB_CLONE_DIR, exist_ok=True)
                _run_git_cmd(["init"], cwd=GITHUB_CLONE_DIR)
                _run_git_cmd(["checkout", "-b", GITHUB_BRANCH], cwd=GITHUB_CLONE_DIR)
                _run_git_cmd(["remote", "add", "origin", remote_url], cwd=GITHUB_CLONE_DIR)
        else:
            ok, out = _run_git_cmd(["pull", "--rebase", "origin", GITHUB_BRANCH],
                                   cwd=GITHUB_CLONE_DIR, secret=token)
            if not ok:
                print(f"⚠️  git pull فشل (هنحاول الـ push عادي):\n{out}")

        _run_git_cmd(["config", "user.email", "results-bot@example.com"], cwd=GITHUB_CLONE_DIR)
        _run_git_cmd(["config", "user.name", "Results Auto-Push"], cwd=GITHUB_CLONE_DIR)

        dst_dir = os.path.join(GITHUB_CLONE_DIR, GITHUB_SUBFOLDER)
        os.makedirs(dst_dir, exist_ok=True)
        for item in os.listdir(WORK_ROOT):
            s = os.path.join(WORK_ROOT, item)
            if os.path.isfile(s) and not item.endswith(".tmp"):
                shutil.copy2(s, os.path.join(dst_dir, item))

        _run_git_cmd(["add", "-A"], cwd=GITHUB_CLONE_DIR)
        ok, out = _run_git_cmd(["commit", "-m", commit_message], cwd=GITHUB_CLONE_DIR)
        if not ok and "nothing to commit" not in out.lower():
            print(f"⚠️  git commit فشل:\n{out}")
            return False

        ok, out = _run_git_cmd(
            ["push", "-u", "origin", f"HEAD:{GITHUB_BRANCH}"],
            cwd=GITHUB_CLONE_DIR,
            env={**os.environ, "GIT_TERMINAL_PROMPT": "0"}, secret=token)
        if not ok:
            print(f"⚠️  git push فشل (التدريب مكمّل، الملفات لسه محلية):\n{out}")
            return False

        print(f"🚀 push ناجح للمستودع {GITHUB_REPO} (فرع {GITHUB_BRANCH}).")
        return True
    except Exception as e:
        print(f"⚠️  GitHub Auto-Push فشل بشكل غير متوقع ({e}) — هنكمّل التدريب.")
        return False


def _load_kaggle_creds_from_colab():
    """لو ضفت KAGGLE_USERNAME و KAGGLE_KEY في Colab Secrets، بيتحطوا كمتغيرات
    بيئة عشان kagglehub يقدر ينزّل الداتاست من غير ما يطلب login."""
    try:
        from google.colab import userdata
        for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
            if not os.environ.get(k):
                try:
                    v = userdata.get(k)
                    if v:
                        os.environ[k] = v
                except Exception:
                    pass
    except Exception:
        pass


_load_kaggle_creds_from_colab()

# استرجاع تلقائي قبل أي حاجة (لو الجلسة اتقطعت أو notebook جديد)
_restore_previous_workdir_from_github()


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Processing on: {device}")
if device == "cpu":
    print("⚠️  مفيش GPU! من Colab: Runtime → Change runtime type → T4 GPU")

# Simple stage-timing registry
STAGE_TIMES = {}

class StageTimer:
    """Context manager that records wall-clock time for a named stage."""
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.time()
        print(f"\n  [{self.name}] started...")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        elapsed = time.time() - self.t0
        STAGE_TIMES[self.name] = elapsed
        mins = elapsed / 60.0
        print(f"  [{self.name}] finished in {elapsed:.1f}s ({mins:.2f} min)")


def print_stage_time_summary():
    if not STAGE_TIMES:
        print("No stage timings recorded yet.")
        return
    print("\n" + "="*40)
    print("  STAGE TIME SUMMARY")
    print("="*40)
    total = 0.0
    for name, secs in STAGE_TIMES.items():
        print(f"  {name:45s} {secs:8.1f}s  ({secs/60:.2f} min)")
        total += secs
    print(f"  {'TOTAL':45s} {total:8.1f}s  ({total/60:.2f} min)")


def count_parameters(model, trainable_only=False):
    if trainable_only:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    return sum(p.numel() for p in model.parameters())


def compute_flops(backbone_module, input_size, device=device):
    """Backbone-only FLOPs/params via thop (feature extraction pass only)."""
    backbone_module = backbone_module.to(device).eval()
    dummy = torch.randn(1, 3, input_size, input_size).to(device)

    class _FeatureOnly(nn.Module):
        def __init__(self, backbone):
            super().__init__()
            self.backbone = backbone

        def forward(self, x):
            x = self.backbone.features(x)
            x = self.backbone.avgpool(x)
            return torch.flatten(x, 1)

    wrapper = _FeatureOnly(backbone_module).to(device).eval()
    with torch.no_grad():
        macs, params = thop_profile(wrapper, inputs=(dummy,), verbose=False)

    gflops = macs / 1e9
    params_m = params / 1e6
    print(f" Backbone profile @ {input_size}x{input_size}: "
          f"{params_m:.2f}M params | {gflops:.4f} GFLOPs (MACs)")
    return {"params": params, "params_M": params_m, "macs": macs, "gflops": gflops}


def report_full_model_stats(model, input_size):
    """Total params of ProtoNet wrapper (backbone only, no extra params) + FLOPs."""
    total_params = count_parameters(model)
    trainable_params = count_parameters(model, trainable_only=True)
    print(f"\n ProtoNet full model: "
          f"{total_params/1e6:.3f}M total params | "
          f"{trainable_params/1e6:.3f}M trainable params")
    flops_info = compute_flops(model.backbone, input_size=input_size)
    return {
        "total_params": total_params,
        "trainable_params": trainable_params,
        **flops_info
    }


def _find_images_folder(root):
    for r, dirs, files in os.walk(root):
        if os.path.basename(r) == "images" and any(
            os.path.isdir(os.path.join(r, d)) for d in dirs
        ):
            return r
    return None


def _find_class_dirs(root):
    """Return a directory whose immediate children are all class-folders
    (heuristic: at least 50 subdirs, each containing images)."""
    for r, dirs, files in os.walk(root):
        if not dirs:
            continue
        sub_dirs = [d for d in dirs if os.path.isdir(os.path.join(r, d))]
        if len(sub_dirs) < 50:  # CUB has 200 classes total / per split >=50
            continue
        sample = os.path.join(r, sub_dirs[0])
        has_imgs = any(f.lower().endswith(('jpg', 'jpeg', 'png'))
                       for f in os.listdir(sample))
        if has_imgs:
            return r
    return None


def prepare_cub200():
    if os.path.exists(CUB_SPLIT_ROOT):
        print(" CUB-200-2011 split already present.")
        return

    print("\n Downloading CUB-200-2011...")
    try:
        path = kagglehub.dataset_download(CUB_KAGGLEHUB_ID)
        dataset_path = path
        print(f" Dataset downloaded to: {dataset_path}")

        os.makedirs(CUB_SPLIT_ROOT, exist_ok=True)

        existing_splits = {}
        for split_name in ["train", "val", "test"]:
            for r, dirs, files in os.walk(dataset_path):
                if os.path.basename(r).lower() == split_name and dirs:
                    existing_splits[split_name] = r
                    break

        if len(existing_splits) == 3:
            print(" Found official train/val/test split folders.")
            for split_name, src_root in existing_splits.items():
                dst_root = os.path.join(CUB_SPLIT_ROOT, split_name)
                if not os.path.exists(dst_root):
                    shutil.copytree(src_root, dst_root)
            print(" CUB-200-2011 Data Structure Ready! (official split found)")
            return

        images_root = _find_images_folder(dataset_path) or _find_class_dirs(dataset_path)
        if images_root is None:
            raise RuntimeError(
                "Could not locate CUB-200-2011 class directories in download."
            )

        print(f" Found images root: {images_root}")
        classes = sorted([d for d in os.listdir(images_root)
                           if os.path.isdir(os.path.join(images_root, d))])
        print(f" Found {len(classes)} classes.")

        rng = random.Random(42)
        rng.shuffle(classes)
        n_total = len(classes)

        if n_total == 200:
            n_train, n_val = 100, 50
        else:
            n_train = int(round(n_total * 0.5))
            n_val   = int(round(n_total * 0.25))
        splits = {
            "train": classes[:n_train],
            "val":   classes[n_train:n_train + n_val],
            "test":  classes[n_train + n_val:],
        }
        print(f"   Split sizes → train: {len(splits['train'])}, "
              f"val: {len(splits['val'])}, test: {len(splits['test'])}")

        for split_name, split_classes in splits.items():
            os.makedirs(os.path.join(CUB_SPLIT_ROOT, split_name), exist_ok=True)
            for c in split_classes:
                src = os.path.join(images_root, c)
                dst = os.path.join(CUB_SPLIT_ROOT, split_name, c)
                if not os.path.exists(dst):
                    shutil.copytree(src, dst)
        print(" CUB-200-2011 Data Structure Ready! (class-disjoint split)")
    except Exception as e:
        print(f" Error downloading/preparing CUB-200-2011: {e}")
        raise


with StageTimer("Stage 0: Data Preparation (CUB-200-2011)"):
    prepare_cub200()

def compute_prototype(z_support):
    """Class prototype = arithmetic mean of raw support embeddings."""
    return z_support.mean(dim=0)


def get_mobilenet_finetuned(num_classes=100, supervised=True):
    print(f" Loading MobileNetV3-LARGE (num_classes={num_classes})...")
    weights = models.MobileNet_V3_Large_Weights.DEFAULT
    model   = models.mobilenet_v3_large(weights=weights)

    if supervised:
        in_features = model.classifier[3].in_features
        model.classifier[3] = nn.Linear(in_features, num_classes)
    else:
        model.classifier = nn.Identity()

    return model


def build_protonet_for_inference(checkpoint_path, num_classes=100):
    base_model = get_mobilenet_finetuned(num_classes=num_classes, supervised=True)
    base_model.classifier = nn.Identity()
    model = MobileProtoNet(base_model)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.to(device)
    model.eval()
    print(f" Loaded checkpoint: {checkpoint_path}")
    return model


class MobileProtoNet(nn.Module):

    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, support_x, query_x, n_way, k_shot):
        x = torch.cat([support_x, query_x], dim=0)
        x = self.backbone.features(x)
        x = self.backbone.avgpool(x)
        z = torch.flatten(x, 1)

        z_support = z[:support_x.size(0)]
        z_query   = z[support_x.size(0):]

        z_proto = torch.stack([
            compute_prototype(z_support[i * k_shot:(i + 1) * k_shot])
            for i in range(n_way)
        ])

        dists = torch.cdist(z_query, z_proto) ** 2
        logits = -dists
        return logits

def get_resize_dim(crop_dim):
    return round(crop_dim * 256 / 224)


def build_transforms(input_size):
    resize_dim = get_resize_dim(input_size)

    train_transform = transforms.Compose([
        transforms.Resize((resize_dim, resize_dim)),
        transforms.RandomCrop(input_size),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.4, 0.4, 0.4),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    test_transform = transforms.Compose([
        transforms.Resize((resize_dim, resize_dim)),
        transforms.CenterCrop(input_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return train_transform, test_transform


CUB_ROOT = CUB_SPLIT_ROOT

BATCH_SIZE_BY_RESOLUTION = {84: 32, 112: 24, 168: 16, 224: 16, 256: 8}


class EpisodicSampler:
    def __init__(self, root, transform):
        self.root      = root
        self.transform = transform
        self.classes   = sorted(os.listdir(root))
        self.data      = {}
        for c in self.classes:
            cls_path = os.path.join(root, c)
            self.data[c] = [
                os.path.join(cls_path, img)
                for img in os.listdir(cls_path)
                if img.lower().endswith(('jpg', 'jpeg', 'png'))
            ]

    def get_episode(self, n_way, k_shot, q_query):
        selected_classes = random.sample(self.classes, n_way)
        support_imgs, query_imgs, query_labels = [], [], []

        for i, cls in enumerate(selected_classes):
            available_imgs = self.data[cls]
            needed = k_shot + q_query
            if len(available_imgs) < needed:
                sampled_paths = random.choices(available_imgs, k=needed)
            else:
                sampled_paths = random.sample(available_imgs, needed)

            for path in sampled_paths[:k_shot]:
                img = Image.open(path).convert('RGB')
                support_imgs.append(self.transform(img))

            for path in sampled_paths[k_shot:]:
                img = Image.open(path).convert('RGB')
                query_imgs.append(self.transform(img))
                query_labels.append(i)

        return (torch.stack(support_imgs),
                torch.stack(query_imgs),
                torch.tensor(query_labels))


def generate_fixed_episodes(root, n_way, k_shot, q_query, n_episodes, seed):
    rng = random.Random(seed)  # RNG محلي مستقل — لا يمس حالة random العامة
    classes = sorted(os.listdir(root))
    data = {}
    for c in classes:
        cls_path = os.path.join(root, c)
        data[c] = [
            os.path.join(cls_path, img)
            for img in os.listdir(cls_path)
            if img.lower().endswith(('jpg', 'jpeg', 'png'))
        ]

    episodes = []
    for _ in range(n_episodes):
        selected_classes = rng.sample(classes, n_way)
        support_paths, query_paths, query_labels = [], [], []

        for i, cls in enumerate(selected_classes):
            available_imgs = data[cls]
            needed = k_shot + q_query
            if len(available_imgs) < needed:
                sampled_paths = rng.choices(available_imgs, k=needed)
            else:
                sampled_paths = rng.sample(available_imgs, needed)

            support_paths.extend(sampled_paths[:k_shot])
            query_paths.extend(sampled_paths[k_shot:])
            query_labels.extend([i] * q_query)

        episodes.append({
            "classes": selected_classes,
            "support_paths": support_paths,
            "query_paths": query_paths,
            "query_labels": query_labels,
        })

    return episodes


def _load_episode_tensors(episode_def, transform, n_way, k_shot):
    support_imgs = [transform(Image.open(p).convert('RGB'))
                     for p in episode_def["support_paths"]]
    query_imgs   = [transform(Image.open(p).convert('RGB'))
                     for p in episode_def["query_paths"]]
    query_labels = torch.tensor(episode_def["query_labels"])
    return torch.stack(support_imgs), torch.stack(query_imgs), query_labels


@torch.no_grad()
def evaluate_fixed_episodes(model, episode_defs, transform, n_way, k_shot, desc=""):

    model.eval()
    accs = []
    for ep_def in tqdm(episode_defs, desc=f"Matched-Eval {desc}", leave=False):
        sx, qx, qy = _load_episode_tensors(ep_def, transform, n_way, k_shot)
        sx, qx, qy = sx.to(device), qx.to(device), qy.to(device)
        logits = model(sx, qx, n_way, k_shot)
        preds  = logits.argmax(dim=1)
        accs.append((preds == qy).float().mean().item())
    return np.array(accs)


def paired_resolution_test(acc_84, acc_224, label=""):
    assert len(acc_84) == len(acc_224), "عدد الـ episodes لازم يتطابق للمقارنة الـ paired"
    delta = acc_224 - acc_84
    mean_delta = delta.mean()
    std_delta  = delta.std(ddof=1)
    n = len(delta)
    se = std_delta / np.sqrt(n)
    ci95 = 1.96 * se

    t_stat, t_pval = scipy_stats.ttest_rel(acc_224, acc_84)
    try:
        w_stat, w_pval = scipy_stats.wilcoxon(acc_224, acc_84)
    except ValueError:
        # Wilcoxon بيفشل لو كل الفروق = صفر
        w_stat, w_pval = float('nan'), float('nan')

    print(f"\n Paired 84x84 vs 224x224 — {label}")
    print(f"   n episodes        : {n}")
    print(f"   Mean Acc @84       : {acc_84.mean():.2%}")
    print(f"   Mean Acc @224      : {acc_224.mean():.2%}")
    print(f"   Mean Δ (224-84)    : {mean_delta:+.2%}  (95% CI ±{ci95:.2%})")
    print(f"   Paired t-test      : t={t_stat:.4f}, p={t_pval:.6f}")
    print(f"   Wilcoxon signed-rank: W={w_stat:.4f}, p={w_pval:.6f}")

    return {
        "n": n,
        "mean_delta": mean_delta,
        "ci95_delta": ci95,
        "t_stat": t_stat,
        "t_pval": t_pval,
        "w_stat": w_stat,
        "w_pval": w_pval,
        "acc_84_mean": acc_84.mean(),
        "acc_224_mean": acc_224.mean(),
    }


def stage1_finetuning(train_loader_simple, epochs=10, num_classes=100):
    print(f"\n STAGE 1: Fine-tuning MobileNet on CUB-200-2011 for {epochs} epochs...")

    model     = get_mobilenet_finetuned(num_classes=num_classes, supervised=True).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9,
                                 nesterov=True, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0

        pbar = tqdm(train_loader_simple, desc=f"Fine-tune Ep {epoch+1}", leave=False)
        for imgs, labels in pbar:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            preds    = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
            pbar.set_postfix({'Acc': f"{correct/total:.1%}"})

        scheduler.step()
        print(f"   Ep {epoch+1}: Loss {total_loss/len(train_loader_simple):.4f} | "
              f"Train Acc {correct/total:.2%}")

    print(" Fine-tuning Complete.")
    return model


def stage2_metatrain(pretrained_state_dict, train_sampler, val_sampler,
                      n_way=5, k_shot=1, episodes=2000, num_classes=100,
                      ckpt_path="best_mobile_proto_cub200.pth"):
    print(f"\n🧠 STAGE 2: Meta-Training (ProtoNet) on CUB-200-2011 "
          f"{n_way}-way {k_shot}-shot...")

    base_model = get_mobilenet_finetuned(num_classes=num_classes, supervised=True)
    base_model.load_state_dict(pretrained_state_dict)
    base_model.classifier = nn.Identity()

    model = MobileProtoNet(base_model).to(device)

    optimizer = torch.optim.Adam(model.backbone.parameters(), lr=1e-5)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    patience     = 0

    for ep in tqdm(range(episodes), desc="Meta-Train"):
        sx, qx, qy = train_sampler.get_episode(n_way, k_shot, q_query=15)
        sx, qx, qy = sx.to(device), qx.to(device), qy.to(device)

        model.train()
        optimizer.zero_grad()
        logits = model(sx, qx, n_way, k_shot)
        loss   = criterion(logits, qy)
        loss.backward()
        optimizer.step()

        if (ep + 1) % 200 == 0:
            val_acc, _ = evaluate(model, val_sampler, n_way, k_shot,
                                   episodes=200, silent=True)
            print(f"\n   Ep {ep+1}: Loss {loss.item():.4f} | Val Acc: {val_acc:.2%}")

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(model.state_dict(), ckpt_path)
                patience = 0
            else:
                patience += 1
                if patience > 10:
                    print(" Early stopping triggered.")
                    break

    return model


@torch.no_grad()
def evaluate(model, sampler, n_way, k_shot, episodes=500, silent=False):
    model.eval()
    accs     = []
    iterator = range(episodes) if silent else tqdm(range(episodes), desc="Evaluating")

    for _ in iterator:
        sx, qx, qy = sampler.get_episode(n_way, k_shot, q_query=15)
        sx, qx, qy = sx.to(device), qx.to(device), qy.to(device)

        logits = model(sx, qx, n_way, k_shot)
        preds  = logits.argmax(dim=1)
        accs.append((preds == qy).float().mean().item())

    mean_acc = np.mean(accs)
    ci95     = 1.96 * np.std(accs) / np.sqrt(episodes)

    if not silent:
        print(f" Result: {mean_acc:.2%} ± {ci95:.2%}")

    return mean_acc, ci95


NUM_TRAIN_CLASSES = 100

def run_experiment(input_size, seed, epochs=10, episodes=2000):
    print("\n" + "#"*60)
    print(f"### RUNNING EXPERIMENT: input_size={input_size}x{input_size} | seed={seed}")
    print("#"*60)

    set_seed(seed)

    train_transform, test_transform = build_transforms(input_size)

    train_dataset_simple = datasets.ImageFolder(
        root=os.path.join(CUB_ROOT, "train"),
        transform=train_transform
    )
    batch_size = BATCH_SIZE_BY_RESOLUTION.get(input_size, 32)
    train_loader_simple = DataLoader(
        train_dataset_simple,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )
    print(f"   Batch size: {batch_size}")

    train_sampler = EpisodicSampler(os.path.join(CUB_ROOT, "train"), test_transform)
    val_sampler   = EpisodicSampler(os.path.join(CUB_ROOT, "val"),   test_transform)
    test_sampler  = EpisodicSampler(os.path.join(CUB_ROOT, "test"),  test_transform)

    tag = f"{input_size}px_seed{seed}"

    backbone_ckpt = os.path.join(WORK_ROOT, f"backbone_pretrained_cub200_{tag}.pth")
    if not os.path.exists(backbone_ckpt):
        with StageTimer(f"Stage 1: Fine-Tuning ({tag})"):
            ft_model = stage1_finetuning(train_loader_simple, epochs=epochs,
                                          num_classes=NUM_TRAIN_CLASSES)
            torch.save(ft_model.state_dict(), backbone_ckpt)
    else:
        print(f" Pre-trained backbone found for {tag}. Skipping Stage 1.")
    pretrained_state_dict = torch.load(backbone_ckpt, map_location=device)

    results = {}

    for k_shot in [1, 5]:
        print("\n" + "="*40)
        print(f" TASK: 5-way {k_shot}-shot ({tag})")
        print("="*40)

        ckpt_path = os.path.join(
            WORK_ROOT, f"best_mobile_proto_cub200_{tag}_{k_shot}shot.pth"
        )

        with StageTimer(f"Stage 2: Meta-Training {k_shot}-shot ({tag})"):
            stage2_metatrain(pretrained_state_dict, train_sampler, val_sampler,
                              n_way=5, k_shot=k_shot, episodes=episodes,
                              num_classes=NUM_TRAIN_CLASSES, ckpt_path=ckpt_path)

        with StageTimer(f"Stage 3: Test Evaluation {k_shot}-shot ({tag})"):
            best_model = build_protonet_for_inference(
                checkpoint_path=ckpt_path, num_classes=NUM_TRAIN_CLASSES
            )
            mean_acc, ci95 = evaluate(best_model, test_sampler, n_way=5, k_shot=k_shot)
            results[k_shot] = {"mean_acc": mean_acc, "ci95": ci95, "ckpt_path": ckpt_path}

        if k_shot == 5:
            print("\n" + "="*40)
            print(f" MODEL SIZE & COMPUTE REPORT ({tag})")
            print("="*40)
            report_full_model_stats(best_model, input_size=input_size)

    return results

PROGRESS_FILE = os.path.join(WORK_ROOT, "progress.json")


def load_progress():

    if not os.path.exists(PROGRESS_FILE):
        return {}, {}
    try:
        with open(PROGRESS_FILE, "r") as f:
            data = json.load(f)
    except (json.JSONDecodeError, OSError) as e:
        print(f"⚠️  ملف التقدّم {PROGRESS_FILE} تالف ({e}) — هنبدأ من الصفر.")
        return {}, {}

    all_results = {}
    for res_str, seeds in data.get("results", {}).items():
        res = int(res_str)
        all_results[res] = {}
        for seed_str, kshots in seeds.items():
            seed = int(seed_str)
            all_results[res][seed] = {}
            for k_str, v in kshots.items():
                all_results[res][seed][int(k_str)] = {
                    "mean_acc": v["mean_acc"],
                    "ci95": v["ci95"],
                    "ckpt_path": v["ckpt_path"],
                }

    matched_accs = {}
    for k_str, seeds in data.get("matched", {}).items():
        k_shot = int(k_str)
        matched_accs[k_shot] = {}
        for seed_str, res_dict in seeds.items():
            seed = int(seed_str)
            matched_accs[k_shot][seed] = {
                84:  np.array(res_dict["84"]),
                224: np.array(res_dict["224"]),
            }

    if all_results or matched_accs:
        print(f"\n  لقينا تقدّم سابق في {PROGRESS_FILE}:")
        for res, seeds in all_results.items():
            print(f"    {res}px: seeds خلّصت = {sorted(seeds.keys())}")
        for k_shot, seeds in matched_accs.items():
            print(f"    Matched-Eval {k_shot}-shot: seeds خلّصت = {sorted(seeds.keys())}")

    return all_results, matched_accs


def save_progress(all_results, matched_accs):

    report = {
        "results": {
            str(res): {
                str(seed): {
                    str(k_shot): {
                        "mean_acc": float(v["mean_acc"]),
                        "ci95": float(v["ci95"]),
                        "ckpt_path": v["ckpt_path"],
                    }
                    for k_shot, v in kshots.items()
                }
                for seed, kshots in seeds.items()
            }
            for res, seeds in all_results.items()
        },
        "matched": {
            str(k_shot): {
                str(seed): {
                    "84":  res_dict[84].tolist(),
                    "224": res_dict[224].tolist(),
                }
                for seed, res_dict in seeds.items()
            }
            for k_shot, seeds in matched_accs.items()
        },
    }
    tmp_path = PROGRESS_FILE + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(report, f, indent=2)
    os.replace(tmp_path, PROGRESS_FILE)


if __name__ == "__main__":
    INPUT_SIZES = [84, 224]
    SEEDS       = [1, 2, 3]
    N_WAY       = 5
    Q_QUERY     = 15
    N_MATCHED_EPISODES = 2000
    MATCHED_EPISODES_SEED = 999

    all_results, matched_accs = load_progress()  # {input_size: {seed: {k_shot: {...}}}}

    for input_size in INPUT_SIZES:
        all_results.setdefault(input_size, {})
        for seed in SEEDS:
            already_done = (
                seed in all_results[input_size]
                and 1 in all_results[input_size][seed]
                and 5 in all_results[input_size][seed]
            )
            if already_done:
                print(f" {input_size}px seed={seed} موجود في التقدّم المحفوظ — تخطّي.")
                continue

            res = run_experiment(input_size, seed, epochs=10, episodes=2000)
            all_results[input_size][seed] = res
            save_progress(all_results, matched_accs)
            # 🆕 push تلقائي لـ GitHub بعد ما كل seed يخلص
            push_results_to_github(
                commit_message=f"Results: {input_size}px seed={seed} done"
            )

    print("\n" + "="*60)
    print(" FINAL SUMMARY — MEAN ± STD OVER 3 SEEDS")
    print("="*60)
    for input_size in INPUT_SIZES:
        for k_shot in [1, 5]:
            accs = [all_results[input_size][s][k_shot]["mean_acc"] for s in SEEDS]
            print(f"  {input_size}x{input_size} | {k_shot}-shot: "
                  f"{np.mean(accs):.2%} ± {np.std(accs):.2%}  "
                  f"(runs: {[f'{a:.2%}' for a in accs]})")

    print("\n" + "="*60)
    print(" MATCHED-EPISODE EVALUATION (paired 84x84 vs 224x224)")
    print("="*60)

    with StageTimer("Stage 4: Matched-Episode Generation"):
        fixed_episodes = {
            k_shot: generate_fixed_episodes(
                root=os.path.join(CUB_ROOT, "test"),
                n_way=N_WAY, k_shot=k_shot, q_query=Q_QUERY,
                n_episodes=N_MATCHED_EPISODES, seed=MATCHED_EPISODES_SEED
            )
            for k_shot in [1, 5]
        }

    _, test_transform_84  = build_transforms(84)
    _, test_transform_224 = build_transforms(224)

    with StageTimer("Stage 5: Matched-Episode Evaluation"):
        for k_shot in [1, 5]:
            matched_accs.setdefault(k_shot, {})
            eps = fixed_episodes[k_shot]

            for seed in SEEDS:
                if seed in matched_accs[k_shot]:
                    print(f" Matched-Eval {k_shot}-shot seed={seed} "
                          f"موجود في التقدّم المحفوظ — تخطّي.")
                    continue

                ckpt_84  = all_results[84][seed][k_shot]["ckpt_path"]
                ckpt_224 = all_results[224][seed][k_shot]["ckpt_path"]

                model_84  = build_protonet_for_inference(ckpt_84,  num_classes=NUM_TRAIN_CLASSES)
                model_224 = build_protonet_for_inference(ckpt_224, num_classes=NUM_TRAIN_CLASSES)

                acc_84  = evaluate_fixed_episodes(model_84,  eps, test_transform_84,
                                                   N_WAY, k_shot, desc=f"84px seed{seed} {k_shot}shot")
                acc_224 = evaluate_fixed_episodes(model_224, eps, test_transform_224,
                                                   N_WAY, k_shot, desc=f"224px seed{seed} {k_shot}shot")

                matched_accs[k_shot][seed] = {84: acc_84, 224: acc_224}
                save_progress(all_results, matched_accs)
                # 🆕 push تلقائي بعد كل matched-eval
                push_results_to_github(
                    commit_message=f"Matched-eval: {k_shot}-shot seed={seed} done"
                )

                paired_resolution_test(acc_84, acc_224,
                                        label=f"{k_shot}-shot, seed={seed}")

            pooled_84  = np.concatenate([matched_accs[k_shot][s][84]  for s in SEEDS])
            pooled_224 = np.concatenate([matched_accs[k_shot][s][224] for s in SEEDS])
            paired_resolution_test(pooled_84, pooled_224,
                                    label=f"{k_shot}-shot, POOLED across {len(SEEDS)} seeds "
                                          f"({N_MATCHED_EPISODES} episodes × {len(SEEDS)} seeds)")

    print_stage_time_summary()
    # 🆕 push نهائي (يشمل أي ملفات اتحدّثت)
    push_results_to_github(commit_message="Final: all experiments done")
    print(f"\n التقدّم والنتائج محفوظة في: {PROGRESS_FILE}")


🔍 بنحاول نسترجع نسخة سابقة من GitHub (Ahmed-Hamam/control84up224, branch=main)...
♻️  لقينا نسخة سابقة في /content/_github_results_repo/cub-Large_results — بننسخها لـ WORK_ROOT...
✅ تم استرجاع 4 ملف — الكود هيكمّل من آخر seed اتسجّل.
 Processing on: cuda

  [Stage 0: Data Preparation (CUB-200-2011)] started...
 CUB-200-2011 split already present.
  [Stage 0: Data Preparation (CUB-200-2011)] finished in 0.0s (0.00 min)

  لقينا تقدّم سابق في /content/cub200_workdir/progress.json:
    84px: seeds خلّصت = [1, 2]
 84px seed=1 موجود في التقدّم المحفوظ — تخطّي.
 84px seed=2 موجود في التقدّم المحفوظ — تخطّي.

############################################################
### RUNNING EXPERIMENT: input_size=84x84 | seed=3
############################################################
   Batch size: 32

  [Stage 1: Fine-Tuning (84px_seed3)] started...

 STAGE 1: Fine-tuning MobileNet on CUB-200-2011 for 10 epochs...
 Loading MobileNetV3-LARGE (num_classes=100)...


   Ep 1: Loss 3.6709 | Train Acc 14.56%


   Ep 2: Loss 2.2177 | Train Acc 39.03%


   Ep 3: Loss 1.6597 | Train Acc 52.05%


   Ep 4: Loss 1.7483 | Train Acc 51.20%


   Ep 5: Loss 1.2475 | Train Acc 62.62%


... (33 outputs left out)


   Ep 600: Loss 1.7298 | Val Acc: 85.69%


Meta-Train:  40%|████      | 800/2000 [11:34<8:42:37, 26.13s/it]


   Ep 800: Loss 0.1316 | Val Acc: 85.46%


Meta-Train:  50%|████▉     | 999/2000 [13:01<06:53,  2.42it/s]